# MixConfig MNIST demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Q9gJYx/MixConfig/blob/main/notebooks/demo_mnist.ipynb) [![ICML 2026](https://img.shields.io/badge/ICML-2026-1d4ed8.svg)](https://icml.cc/virtual/2026/poster/63010) [![arXiv](https://img.shields.io/badge/arXiv-2510.19248-b31b1b.svg)](https://arxiv.org/abs/2510.19248) [![OpenReview](https://img.shields.io/badge/OpenReview-aw6alulxr8-8c1b13.svg)](https://openreview.net/forum?id=aw6alulxr8)

Smoke test for the pre-extracted MixConfig artifacts on MNIST-70k.

**What this notebook does:**
1. Loads `data/mnist_configs.npz` (configurations + energy statistics from BlueRed front + HOG features).
2. Validates the artifact via `src.mixconfig.validate_configurations`.
3. Inspects the BlueRed-front structure (9 stable configurations spanning k=1 coarse to k=70000 singletons; the highest-resolution non-trivial configuration is k=9 with persistence λ ≈ 0.957, which approximates but does not exactly match the 10 MNIST digit classes - the bluered 10-cluster match is the n=5000 subset, not the 70k variant we ship here).
4. Builds the Energy-Aware Selector and runs a forward pass on a 1000-sample subset, dropping the singleton-endpoint configuration.

Runtime: under 60 seconds on Colab CPU. No GPU needed for this smoke test.

For full reproduction commands across tabular / vision / molecular / text benchmarks, see [README.md](https://github.com/Q9gJYx/MixConfig/blob/main/README.md#reproducing-paper-results).

In [1]:
# Colab setup: clone the repo and put it on the Python path.
# Local execution: skip this cell (the parent dir is already on the path).
import os, sys
if not os.path.exists("MixConfig") and not os.path.exists("../src/mixconfig"):
    !git clone -q https://github.com/Q9gJYx/MixConfig.git
REPO = "MixConfig" if os.path.exists("MixConfig") else ".."
sys.path.insert(0, REPO)
print(f"Using repo root: {REPO}")

Using repo root: ..


In [2]:
import numpy as np

bundle = np.load(f"{REPO}/data/mnist_configs.npz")
print("Keys in the bundle:")
for k in bundle.files:
    print(f"  {k}: shape={bundle[k].shape} dtype={bundle[k].dtype}")

Keys in the bundle:
  configs: shape=(70000, 9) dtype=int64
  energy_stats: shape=(9, 4) dtype=float32
  labels: shape=(70000,) dtype=int64
  bluered_lambda: shape=(9,) dtype=float64
  bluered_mu: shape=(9,) dtype=float64
  bluered_gamma_rng: shape=(9, 2) dtype=float64


In [3]:
from src.mixconfig import validate_configurations

configs = bundle["configs"]
energy_stats = bundle["energy_stats"]
max_id = validate_configurations(configs, energy_stats)
n_clusters = [int(len(np.unique(configs[:, j]))) for j in range(configs.shape[1])]

print(f"validate_configurations: OK  (max cluster id = {max_id})")
print(f"BlueRed front: {configs.shape[1]} configurations on n={configs.shape[0]:,} samples")
print(f"Cluster counts per config: {n_clusters}\n")
print("Energy statistics [H, h_a, h_r, delta_gamma]:")
with np.printoptions(precision=3, suppress=True):
    print(energy_stats)

# Drop the singleton-endpoint configuration (k = n samples) before feeding the
# selector: ClusterAssignmentEmbedder allocates max_clusters rows per configuration.
keep = configs.shape[1] - 1
configs_kept = configs[:, :keep]
energy_kept = energy_stats[:keep]
max_k = int(configs_kept.max()) + 1
print(f"\nKeeping {keep}/{configs.shape[1]} configs for the selector (max cluster id = {max_k - 1}).")

validate_configurations: OK  (max cluster id = 70000)
BlueRed front: 9 configurations on n=70,000 samples
Cluster counts per config: [1, 2, 3, 4, 9, 12, 47, 191, 70000]

Energy statistics [H, h_a, h_r, delta_gamma]:
[[ -0.      2.016   0.      0.007]
 [  0.345   1.858   1.682   0.006]
 [  0.915   1.849   1.492   0.01 ]
 [  1.292   1.815   1.499   0.053]
 [  2.157   1.709   1.586   0.154]
 [  2.396   1.656   1.663   1.064]
 [  3.619   1.492   1.962   4.398]
 [  5.101   1.398   2.011 108.181]
 [ 11.156   0.      2.831 216.362]]

Keeping 8/9 configs for the selector (max cluster id = 191).


## End-to-end MNIST classification

The cells above only verify the *artifact*. The cells below train a small classifier head end-to-end (configurations → Energy-Aware Selector → mixed representation → linear probe) and compare to the two baselines from the paper's headline framing:

- **Single-config** — use the embedding of one configuration column only (`k=9`, the highest-resolution non-trivial config). No mixing.
- **Uniform-mix** — average the embeddings of all kept configurations with equal weights. Static mixing, no learned selector.

5 000 / 1 000 train/val split on raw 784-dim MNIST pixels, 5 epochs, Adam(1e-3), batch 256 — runs in under three minutes on Colab CPU. Headline benchmark numbers are in the paper; this is a sanity check that the released artifact and selector wire up correctly.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from src.mixconfig import EnergyAwareSelector, ClusterAssignmentEmbedder

torch.manual_seed(0)
np.random.seed(0)

print("Fetching MNIST features (cached on second run)...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="liac-arff")

# 5000 train / 1000 val on the first 6000 MNIST samples (deterministic order).
n_demo = 6000
X = mnist.data[:n_demo].astype(np.float32) / 255.0
y = bundle["labels"][:n_demo].astype(np.int64)
c = configs_kept[:n_demo].astype(np.int64)

X_tr_t, X_va_t = torch.tensor(X[:5000]), torch.tensor(X[5000:])
y_tr_t, y_va_t = torch.tensor(y[:5000]), torch.tensor(y[5000:])
c_tr_t, c_va_t = torch.tensor(c[:5000]), torch.tensor(c[5000:])
es_t = torch.tensor(energy_kept, dtype=torch.float32)


def train_eval(model: nn.Module, label: str, n_epochs: int = 5, batch_size: int = 256) -> float:
    """Train `model` on the MNIST split above; return final validation accuracy."""
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    n_train = X_tr_t.shape[0]
    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_train)
        losses = []
        for i in range(0, n_train, batch_size):
            idx = perm[i : i + batch_size]
            loss = crit(model(X_tr_t[idx], c_tr_t[idx], es_t), y_tr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            va_acc = (model(X_va_t, c_va_t, es_t).argmax(-1) == y_va_t).float().mean().item()
        print(f"  {label} epoch {epoch+1}: train_loss={np.mean(losses):.4f}  val_acc={va_acc:.4f}")
    return va_acc


class MixConfigClassifier(nn.Module):
    """Energy-Aware Selector + linear classifier head."""

    def __init__(self):
        super().__init__()
        self.selector = EnergyAwareSelector(
            input_dim=784, n_configs=keep, max_clusters=max_k,
            context_dim=64, cluster_embed_dim=32,
        )
        self.head = nn.Linear(32, 10)

    def forward(self, x, c, e):
        return self.head(self.selector.get_mixed_representation(x, c, e))


print("\nTraining MixConfig (learned per-sample selector weights)")
torch.manual_seed(0)
mc_acc = train_eval(MixConfigClassifier(), label="MixConfig")

Fetching MNIST features (cached on second run)...



Training MixConfig (learned per-sample selector weights)


  MixConfig epoch 1: train_loss=2.2988  val_acc=0.2210


  MixConfig epoch 2: train_loss=2.2321  val_acc=0.6030


  MixConfig epoch 3: train_loss=2.0936  val_acc=0.9580


  MixConfig epoch 4: train_loss=1.8308  val_acc=0.9720


  MixConfig epoch 5: train_loss=1.4294  val_acc=0.9720


In [5]:
class SingleConfigBaseline(nn.Module):
    """Use one configuration column, no mixing."""

    def __init__(self, config_idx: int):
        super().__init__()
        self.config_idx = config_idx
        self.embed = nn.Embedding(max_k, 32)
        self.head = nn.Linear(32, 10)

    def forward(self, x, c, e):
        return self.head(self.embed(c[:, self.config_idx]))


class UniformMixBaseline(nn.Module):
    """Average all kept-config embeddings with equal weights."""

    def __init__(self):
        super().__init__()
        self.embedder = ClusterAssignmentEmbedder(n_configs=keep, max_clusters=max_k, embed_dim=32)
        self.head = nn.Linear(32, 10)

    def forward(self, x, c, e):
        return self.head(self.embedder(c).mean(dim=1))


print("Training single-config baseline (k=9 column)")
torch.manual_seed(0)
sc_acc = train_eval(SingleConfigBaseline(config_idx=4), label="single-config")

print("\nTraining uniform-mix baseline")
torch.manual_seed(0)
um_acc = train_eval(UniformMixBaseline(), label="uniform-mix")

print("\n=== MNIST 5k/1k val accuracy ===")
print(f"  Single-config (k=9 only):     {sc_acc:.4f}")
print(f"  Uniform mix (all 8 configs):  {um_acc:.4f}")
print(f"  MixConfig (learned weights):  {mc_acc:.4f}")
print(f"\n  MixConfig vs uniform-mix:    {(mc_acc - um_acc) * 100:+.2f} pts")
print(f"  MixConfig vs single-config:  {(mc_acc - sc_acc) * 100:+.2f} pts")

Training single-config baseline (k=9 column)
  single-config epoch 1: train_loss=2.1382  val_acc=0.4910
  single-config epoch 2: train_loss=1.7370  val_acc=0.6030
  single-config epoch 3: train_loss=1.4051  val_acc=0.7870
  single-config epoch 4: train_loss=1.1382  val_acc=0.7870
  single-config epoch 5: train_loss=0.9267  val_acc=0.8770

Training uniform-mix baseline
  uniform-mix epoch 1: train_loss=2.2926  val_acc=0.1920


  uniform-mix epoch 2: train_loss=2.2485  val_acc=0.3250
  uniform-mix epoch 3: train_loss=2.1835  val_acc=0.8850
  uniform-mix epoch 4: train_loss=2.0906  val_acc=0.8850
  uniform-mix epoch 5: train_loss=1.9703  val_acc=0.9690

=== MNIST 5k/1k val accuracy ===
  Single-config (k=9 only):     0.8770
  Uniform mix (all 8 configs):  0.9690
  MixConfig (learned weights):  0.9720

  MixConfig vs uniform-mix:    +0.30 pts
  MixConfig vs single-config:  +9.50 pts


## Next steps

- The headline benchmarks (OpenML-CC18, CIFAR-100 / ImageNet-1K linear probe on CLIP, OGBG-MolHIV / BBBP / BACE / QM9, SST-2 / AG News) live under [`experiments/`](https://github.com/Q9gJYx/MixConfig/tree/main/experiments). Each script supports `--mode {base, +config}` for the single-resolution / static-mixing / MixConfig comparison from the paper.
- Swap the raw 784-dim MNIST pixels above for any frozen embedding (CLIP, BERT, GIN) without re-extracting configurations. The selector's `EnergyStatistics` module computes feature-space H/h_a/h_r on the fly when you pass `energy_stats=None`; `delta_gamma` is already pre-baked from the upstream BlueRed γ-band widths.
- For the substitution interface (use your own multi-resolution clustering routine), see [`docs/configurations.md`](https://github.com/Q9gJYx/MixConfig/blob/main/docs/configurations.md). For the upstream Parallel-DT / BlueRed status, see [`docs/dependencies.md`](https://github.com/Q9gJYx/MixConfig/blob/main/docs/dependencies.md).